# HoopStats on Colab (GPU) — SAM2 tracking + portfolio report

Runs the `hoopstats` pipeline on a GPU so the court map uses **SAM2 mask
propagation** (notebook-quality tracking) instead of CPU ByteTrack, then builds
the self-contained portfolio page.

**Before you start:** Runtime → Change runtime type → **GPU** (T4 is fine).
Add your keys under the Colab **Secrets** (🔑) panel: `ROBOFLOW_API_KEY` and `HF_TOKEN`.


## 1. API keys + GPU check


In [ ]:
import os
# reduce CUDA fragmentation (must be set before torch initializes CUDA)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
from google.colab import userdata
os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
assert torch.cuda.is_available(), \n    'No GPU. Runtime > Change runtime type > GPU, then rerun.'
print('keys set; GPU:', torch.cuda.get_device_name(0))


## 2. Get the hoopstats repo

Either clone from GitHub (set `REPO_URL`) or mount Google Drive and point
`PROJECT_DIR` at a copy you uploaded.


In [ ]:
REPO_URL = ''  # TODO: e.g. 'https://github.com/youruser/basketball-box-score-ai.git'
PROJECT_DIR = '/content/basketball-box-score-ai'

if REPO_URL:
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    from google.colab import drive; drive.mount('/content/drive')
    # TODO: set PROJECT_DIR to your repo path under /content/drive/MyDrive/...
%cd {PROJECT_DIR}


## 3. Install SAM2 real-time + checkpoints

Same fork the reference notebook uses; it provides `build_sam2_camera_predictor`.
`--no-build-isolation` lets the build see Colab's torch (its setup.py imports torch
at build time), and `SAM2_BUILD_CUDA=0` skips the optional CUDA extension that fails
to compile — the camera predictor does not need it.


In [ ]:
%cd /content
!git clone https://github.com/Gy920/segment-anything-2-real-time.git
%cd /content/segment-anything-2-real-time
!SAM2_BUILD_CUDA=0 pip install -e . --no-build-isolation -q
!(cd checkpoints && bash download_ckpts.sh)
SAM2_DIR = '/content/segment-anything-2-real-time'
# sanity: the import that --tracker sam2 needs
from sam2.build_sam import build_sam2_camera_predictor
print('SAM2 import OK')


## 4. Install hoopstats runtime deps (GPU)

Install the package without its CPU deps, then the GPU/runtime deps the tracking
+ report paths actually use (skips paddleocr, not needed here).


In [ ]:
%cd {PROJECT_DIR}
%pip install -e . --no-deps -q
%pip install -q inference-gpu 'supervision==0.27.0rc4' \
  'git+https://github.com/roboflow/sports.git@feat/basketball' \
  scipy opencv-python python-dotenv tqdm


## 5. Point at SAM2 checkpoint/config and your clip

Defaults to the **small** SAM2 model, which fits a free-Colab T4. The large
model OOMs on a T4 — only switch to `base_plus`/`large` on a bigger GPU.

Use a **short hero clip** (one possession). SAM2 prompts the players on frame 0
and propagates — substitutions / hard camera cuts break that, so keep it short.


In [ ]:
# small fits a free-Colab T4 (~15GB). Use base_plus/large only on bigger GPUs (A100).
os.environ['SAM2_CHECKPOINT'] = f'{SAM2_DIR}/checkpoints/sam2.1_hiera_small.pt'
os.environ['SAM2_CONFIG'] = 'configs/sam2.1/sam2.1_hiera_s.yaml'  # Hydra config name

VIDEO = '/content/clip.mp4'  # TODO: upload your clip or point at Drive
FRAME = 0                     # frame for the still images (clean half-court view)
assert os.path.exists(VIDEO), f'VIDEO not found: {VIDEO}'


## 6. SAM2 court mapping (the hero clip)

`--tracker sam2` runs SAM2 propagation + `clean_paths`. Drop `--max-frames` to do
the whole clip. Watch the output for `Saved court map video`.


In [ ]:
!hoopstats map-court-video --video {VIDEO} --out data/outputs --debug \
  --tracker sam2 \
  --sam2-checkpoint $SAM2_CHECKPOINT --sam2-config $SAM2_CONFIG


## 7. Stage stills for the report


In [ ]:
!hoopstats detect-frame     --video {VIDEO} --out data/outputs --frame {FRAME}
!hoopstats detect-keypoints --video {VIDEO} --out data/outputs --frame {FRAME}
!hoopstats map-court        --video {VIDEO} --out data/outputs --frame {FRAME}


## 8. Build the portfolio report


In [ ]:
stem = os.path.splitext(os.path.basename(VIDEO))[0]
court = f'data/outputs/court_map_video/{stem}-court-map.mp4'
debug = f'data/outputs/court_map_video/{stem}-tracking-debug.mp4'
!hoopstats report --frames-dir data/outputs \
  --court-video {court} --debug-video {debug} \
  --out data/portfolio/pipeline.html

# Show exactly what was produced and where
!ls -la data/portfolio data/portfolio/assets 2>/dev/null
print('PORTFOLIO HTML:', os.path.abspath('data/portfolio/pipeline.html'),
      '->', os.path.exists('data/portfolio/pipeline.html'))


## 9. Download the portfolio (HTML + assets)


In [ ]:
import shutil
assert os.path.exists('data/portfolio/pipeline.html'), \
    'No portfolio yet — check the report cell output above for errors.'
shutil.make_archive('/content/hoopstats_portfolio', 'zip', 'data/portfolio')
from google.colab import files
files.download('/content/hoopstats_portfolio.zip')


### Notes

- The portfolio is written to `PROJECT_DIR/data/portfolio/` (inside the cloned
  repo), **not** the top-level `/content`. Use cell 9 to download it.
- If SAM2 errors on the config path, it's Hydra resolution: keep the config as the
  package-relative name (as above), or `%cd` into the SAM2 repo first.
- For **whole-game** stats (shots/box score) stay on CPU ByteTrack via
  `process-game`; SAM2 here is for the short tracking showcase.
